# Giai đoạn 3–4 — 12 cấu hình + GridSearchCV

Theo kế hoạch dự án v3.1. Nhãn `recommended_career` mô tả **nhóm nghề nghiệp được khuyến nghị**, không phải tỷ lệ trúng tuyển đại học.

In [ ]:
from pathlib import Path
import os, sys
ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
os.chdir(ROOT)
print("Project:", ROOT)

## 1. Ma trận 3 × 4 kiểu chọn đặc trưng

In [ ]:
import pandas as pd
from career_ml.config import MODEL_NAMES,FS_NAMES
from career_ml.modeling import param_grid
matrix=pd.DataFrame([{"model":m,"fs":fs,"grid_quick":str(param_grid(m,fs,"quick"))}
                     for m in MODEL_NAMES for fs in FS_NAMES])
display(matrix)

## 2. Chạy tất cả thực nghiệm

**Cảnh báo:** Có thể mất nhiều phút đến vài giờ tùy máy. Dữ liệu đầy đủ 12.000 dòng, 5 folds, gồm 12 cấu hình. Chọn `quick` để chạy lưới nhỏ 2 ứng viên/tổ hợp; `standard` hoặc `full` cho grid rộng hơn.

In [ ]:
from career_ml.experiment import run_experiment
results = run_experiment(preset="quick",jobs=2,compare_calibration=False)
display(pd.DataFrame(results).sort_values("best_cv_f1_macro",ascending=False))

## 3. Xem kết quả lựa chọn bằng CV và số đặc trưng được chọn

In [ ]:
from career_ml.config import OUTPUT
import json
with open(OUTPUT / "tables" / "selected_model_train_cv.json",encoding="utf-8") as f:
    chosen=json.load(f)
print(chosen)
results=pd.read_csv(OUTPUT / "tables" / "cv_experiment_results.csv",keep_default_na=False)
display(results[["model","fs","best_cv_f1_macro","std_cv_f1_macro","num_selected_features","seconds"]])

**Quy tắc:** Đã chọn xong bằng Train CV, không quay lại chỉnh mô hình dựa trên Test. Nếu cần hiệu chỉnh xác suất thì chọn bằng validation thuộc Train (`--compare-calibration`) chứ không dùng Test.